In [1]:
%load_ext autoreload
# autoreload 2：改了 benchmark/ 下的文件不用重启 kernel
%autoreload 2

import os
os.chdir(os.path.expanduser("~/projects/LLM/hw2"))   # benchmark 包以 hw2/ 为根

import torch
import pandas as pd
from benchmark import attention_sweep, bench_attention
from cs336_basics.model import scaled_dot_product_attention as sdpa   # §4.1 的被测对象

torch.backends.cuda.matmul.allow_tf32 = False        # fp32 基准：不偷偷走 tensor core
torch.backends.cudnn.allow_tf32 = False

PROF = "profiles"                       # nsys 报告、显存快照（gitignored）
def assets(sec):                        # 每节的落盘目录 notes/assets/s<sec>；各节开头 OUT = assets(n)
    d = f"notes/assets/s{sec}"; os.makedirs(d, exist_ok=True); return d

print(torch.cuda.get_device_name(0), torch.__version__)

NVIDIA GeForce RTX 5090 2.11.0+cu130


## 4 GPU Kernels

PDF §4。编号按 problem 顺序（4.1 `pytorch_attention` = PDF 4.1.1，4.2 `torch_compile` = PDF 4.2，
4.3–4.5 三题在 PDF 里都归在 4.2.2 下）。

In [2]:
OUT = assets(4)

### 4.1 `pytorch_attention`

被测的是 hw1 的 `scaled_dot_product_attention(q, k, v, mask=None)`，
输入 `[batch, seq, d]` —— batch 固定 8，**去掉 head 维**（题面 (i)）。

每个 (d, seq) 格子量五个数：100 次前向 ms、反向开始前的 `memory_allocated()`、100 次反向 ms，外加前向峰值和全程峰值 `max_memory_allocated()`；
OOM 的格子记 `status`，不中断 sweep。

In [3]:
# 实验网格（题面 (ii)）：4 × 5 = 20 格。大格子预期 OOM —— 那正是题目要的答案
D_MODELS = [16, 32, 64, 128]
SEQ_LENS = [256, 1024, 4096, 8192, 16384]

df = attention_sweep(sdpa, D_MODELS, SEQ_LENS, f"{OUT}/attention_fp32.md")
df

[1/20] d=16 seq=256

[20/20] d=128 seq=16384


,d,seq,fwd_ms,bwd_ms,mem_before_bwd_gib,peak_fwd_gib,peak_gib,status
0,16,256,0.084918,0.211320,0.020287,0.024078,0.028466,ok
1,16,1024,0.174846,0.524524,0.080414,0.142456,0.206880,ok
2,16,4096,4.558249,11.346850,1.024048,2.022217,3.029908,ok
3,16,8192,18.063294,44.782066,4.032227,8.028564,12.043946,ok
4,16,16384,NaN,NaN,NaN,NaN,NaN,OOM@warmup
5,32,256,0.082538,0.263421,0.020775,0.024445,0.029321,ok
6,32,1024,0.174076,0.524068,0.082367,0.143921,0.210298,ok
7,32,4096,4.623469,11.435885,1.031860,2.028076,3.043580,ok
8,32,8192,18.138672,44.923645,4.047852,8.040283,12.071290,ok
9,32,16384,NaN,NaN,NaN,NaN,NaN,OOM@warmup


#### 表 4.1-1 / 4.1-2

`assets/s4/tables_4_1.py`：表 4.1-1 是 sweep 的透视；`forward_steps(8192)` 逐步跑一遍 sdpa 前向、每步读 `memory_allocated()`，
给出表 4.1-2「前向峰值 4 份」的依据，seq 16384 列按 ×4 外推（它本身跑不到第 4 步）。

In [4]:
import runpy
T = runpy.run_path(f"{OUT}/tables_4_1.py")
t1, t2 = T["pivot_tables"](df)
steps = T["forward_steps"](8192); steps["seq 16384 外推 GiB"] = steps.iloc[:, 2] * 4
display(t1, t2, steps)

seq,256,1024,4096,8192,16384
d,,,,,
16,0.08 / 0.21,0.17 / 0.52,4.56 / 11.35,18.06 / 44.78,OOM
32,0.08 / 0.26,0.17 / 0.52,4.62 / 11.44,18.14 / 44.92,OOM
64,0.08 / 0.24,0.18 / 0.50,4.88 / 12.25,19.75 / 47.54,OOM
128,0.08 / 0.26,0.26 / 0.69,6.07 / 14.12,23.92 / 54.57,OOM


seq,256,1024,4096,8192,16384
反向前 memory_allocated,0.020,0.080,1.024,4.032,OOM（估 16.0）
前向峰值,0.024,0.142,2.022,8.029,OOM（估 32.0）
前向+反向峰值,0.028,0.207,3.030,12.044,OOM（估 48.0）


,前向步骤,新增张量,seq 8192 活着的 GiB,seq 16384 外推 GiB
0,S = QKᵀ/√d,S,2.000000,8.000000
1,x − max,x_shifted,4.000732,16.002930
2,exp(x_shifted),nume,6.000732,24.002930
3,P = nume / sum,P,8.000977,32.003906
4,函数返回（释放 S、x_shifted）,,4.004639,16.018555


#### 两个验证

1. seq 16384 炸的时候显存里已经有多少——预期 3 份 = 24 GiB，第 4 份申请失败。
2. 换成融合的 `torch.softmax`（只有 S、P 两份），同一格前向能不能过——预期能，峰值 16 GiB。

In [5]:
GiB = 2**30
Q, K, V = (torch.randn(8, 16384, 16, device="cuda", requires_grad=True) for _ in range(3))

# 1. hw1 手写 softmax：炸在第 4 份
torch.cuda.reset_peak_memory_stats()
try:
    O = sdpa(Q, K, V); torch.cuda.synchronize(); print("hw1 版前向过了？", torch.cuda.max_memory_allocated() / GiB)
except torch.cuda.OutOfMemoryError:
    print(f"hw1 版前向 OOM：炸时已分配 {torch.cuda.memory_allocated() / GiB:.1f} GiB")
torch.cuda.empty_cache()

# 2. 融合 softmax：只有 S 和 P
import math
from einops import einsum
torch.cuda.reset_peak_memory_stats()
S = einsum(Q, K, "b q d, b k d -> b q k") / math.sqrt(16)
P = torch.softmax(S, dim=-1)
O = einsum(P, V, "b q k, b k d -> b q d"); torch.cuda.synchronize()
print(f"torch.softmax 版前向 OK：峰值 {torch.cuda.max_memory_allocated() / GiB:.2f} GiB")
del Q, K, V, S, P, O; torch.cuda.empty_cache()

hw1 版前向 OOM：炸时已分配 24.0 GiB
torch.softmax 版前向 OK：峰值 16.05 GiB


### 4.2 `torch_compile`

(a) 同一份 sdpa 用 `torch.compile` 包一层，网格、预热、步数和 §4.1 完全一致，对比 fwd / bwd。
(b) 整个 Transformer `torch.compile(model)`，走第一篇 §2.1 的整模型 benchmark（`BenchConfig(compile=True)`），
三档 × forward / fwd_bwd / full，对比 eager。

预热是关键：`torch.compile` 的编译发生在**第一次调用**，秒级；第一篇 §2.1(c) 说的「eager 一步就稳、compile 留到 §4.2 看」就是这里。
先用 `warmup=0` 看首步开销，再决定正式表用几步预热。

In [6]:
# (a) 编译版 attention：被测函数就是 sdpa 包一层 torch.compile，其余和 §4.1 一模一样
sdpa_compiled = torch.compile(sdpa)

df_c = attention_sweep(sdpa_compiled, D_MODELS, SEQ_LENS, f"{OUT}/attention_compiled.md")

# eager / compiled 并排 + 加速比
cmp = df[["d", "seq", "fwd_ms", "bwd_ms", "peak_fwd_gib", "status"]].merge(
    df_c[["d", "seq", "fwd_ms", "bwd_ms", "peak_fwd_gib", "status"]], on=["d", "seq"], suffixes=("_eager", "_compiled"))
cmp["fwd_speedup"] = cmp.fwd_ms_eager / cmp.fwd_ms_compiled
cmp["bwd_speedup"] = cmp.bwd_ms_eager / cmp.bwd_ms_compiled
cmp.to_json(f"{OUT}/attention_eager_vs_compiled.json", orient="records", indent=1)
cmp

[1/20] d=16 seq=256

/home/yc/projects/LLM/hw2/.venv/lib/python3.13/site-packages/torch/_inductor/compile_fx.py:322: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(


[20/20] d=128 seq=16384


,d,seq,fwd_ms_eager,bwd_ms_eager,peak_fwd_gib_eager,status_eager,fwd_ms_compiled,bwd_ms_compiled,peak_fwd_gib_compiled,status_compiled,fwd_speedup,bwd_speedup
0,16,256,0.084918,0.211320,0.024078,ok,0.068700,0.142778,0.020294,ok,1.236073,1.480063
1,16,1024,0.174846,0.524524,0.142456,ok,0.133014,0.315929,0.080444,ok,1.314498,1.660257
2,16,4096,4.558249,11.346850,2.022217,ok,1.699574,4.596663,1.024170,ok,2.681994,2.468497
3,16,8192,18.063294,44.782066,8.028564,ok,7.023332,17.224163,4.032471,ok,2.571898,2.599956
4,16,16384,NaN,NaN,NaN,OOM@warmup,NaN,NaN,NaN,OOM@warmup,NaN,NaN
5,32,256,0.082538,0.263421,0.024445,ok,0.079010,0.138552,0.020782,ok,1.044664,1.901236
6,32,1024,0.174076,0.524068,0.143921,ok,0.122178,0.300508,0.082397,ok,1.424769,1.743942
7,32,4096,4.623469,11.435885,2.028076,ok,2.105883,4.639841,1.031982,ok,2.195501,2.464715
8,32,8192,18.138672,44.923645,8.040283,ok,8.866186,18.676065,4.048096,ok,2.045826,2.405413
9,32,16384,NaN,NaN,NaN,OOM@warmup,NaN,NaN,NaN,OOM@warmup,NaN,NaN


In [7]:
# (b-0) 首步开销：warmup=0，看 torch.compile 第一步花多久。
# 必须 isolate：每个配置起独立子进程，且 compile 的子进程把 inductor / triton 缓存指到临时目录（sweep._run_isolated），
# 否则同 kernel 里第二次跑、甚至换进程都会命中缓存，首步就不是冷编译了
from benchmark import BenchConfig, sweep
first = sweep([BenchConfig("small", 512, "full", warmup=0, steps=5, compile=c) for c in (False, True)],
              f"{OUT}/compile_first_step.md", isolate=True, as_frame=True)
first[["compile", "first_ms", "rest_avg_ms", "avg_ms", "status"]]

已写出：notes/assets/s4/compile_first_step.md                   
         notes/assets/s4/compile_first_step.json（含逐步原始耗时）


,compile,first_ms,rest_avg_ms,avg_ms,status
0,False,429.14,57.26,131.64,OK
1,True,14458.35,46.99,2929.26,OK


In [8]:
# (b) 整个 Transformer：eager vs torch.compile(model)，三档 × 三模式（batch 4，seq 512，同第一篇 §2.1）。
# 同样 isolate：每个配置冷编译，warmup=5 把编译吃掉后再计时
cfgs = [BenchConfig(size, 512, mode, warmup=5, steps=10, compile=c)
        for size in ("small", "medium", "large") for mode in ("forward", "fwd_bwd", "full") for c in (False, True)]
model_cmp = sweep(cfgs, f"{OUT}/model_eager_vs_compiled.md", isolate=True, as_frame=True)
model_cmp.pivot(index=["size", "mode"], columns="compile", values=["avg_ms", "peak_mem_gib"])

已写出：notes/assets/s4/model_eager_vs_compiled.md              
         notes/assets/s4/model_eager_vs_compiled.json（含逐步原始耗时）


avg_ms         peak_mem_gib       
compile          False   True         False  True 
size   mode                                       
large  forward  117.16   95.29        20.18  16.65
       full        NaN  293.00        27.41  23.97
       fwd_bwd  374.88  271.01        20.28  16.75
medium forward   48.23   42.02        10.48   8.59
       full     165.23  131.74        13.74  11.85
       fwd_bwd  154.37  123.01        10.58   8.69
small  forward   16.53   14.07         3.98   3.26
       full      53.80   43.85         5.04   4.33
       fwd_bwd   49.82   40.72         4.08   3.37

# 5 Triton

In [10]:
# 5.0 热身：PDF §4.2.1 weighted sum —— eager / torch.compile / Triton 对比
from triton.testing import do_bench
from cs336_systems.weightsum_example import WeightedSumFunc, weighted_sum

x = torch.randn(8192, 1024, device="cuda", requires_grad=True)
w = torch.randn(1024, device="cuda", requires_grad=True)

# 正确性：Triton 前向对上 eager
print("max|diff| =", (WeightedSumFunc.apply(x, w) - weighted_sum(x, w)).abs().max().item())

# 计时：do_bench 自带预热，返回中位数 ms
impls = {"eager": weighted_sum, "compiled": torch.compile(weighted_sum), "triton": WeightedSumFunc.apply}
pd.DataFrame({name: {"fwd_ms": do_bench(lambda: f(x, w)),
                     "fwd_bwd_ms": do_bench(lambda: f(x, w).sum().backward())}
              for name, f in impls.items()})

max|diff| = 2.288818359375e-05


,eager,compiled,triton
fwd_ms,0.056536,0.028043,0.030074
fwd_bwd_ms,0.165846,0.094335,0.115272
